# R01 — Catalogue audit

Status: **executed**. This notebook audits one real, provenance-unconfirmed supplier workbook, per [ADR-0005](../../adr/ADR-0005-catalogue-workbook-provenance-and-oq011.md), which is the ruling that unblocks this notebook and is authoritative over any summary below.

**Every finding in this notebook is scoped to the pinned file and its md5 hash below — never to "the Curalina catalogue."** A different export of the same supplier relationship would need its own R01 run.

## 0. Manifest

Run ID, git SHA, seed, package versions, and — in place of a `rules_version`/`snapshot_id` (neither applies to an audit-only run) — the md5 of every input workbook this run pins.

In [1]:
import json
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = Path("runs") / f"R01_{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
SEED = 20260913

import random
random.seed(SEED)  # no model randomness in this notebook; recorded for the standard's sake

def _pkg_version(name):
    try:
        from importlib.metadata import version
        return version(name)
    except Exception:
        return None

manifest = {
    "run_id": RUN_ID,
    "notebook": "R01_catalogue_audit",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "git_sha": subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip() or "uncommitted",
    "seed": SEED,
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "rules_version": None,  # not applicable: this is a source-audit run, not a rules-engine run
    "snapshot_id": None,    # filled in Section 3 once per-supplier snapshots are built
    "packages": {
        "openpyxl": _pkg_version("openpyxl"),
        "curalina-recommendation": _pkg_version("curalina-recommendation"),
    },
    "pinned_inputs": {},  # filled in Section 2 with each workbook's md5
}
manifest

{'run_id': '20260913T180508Z',
 'notebook': 'R01_catalogue_audit',
 'timestamp_utc': '2026-09-13T18:05:08.529393+00:00',
 'git_sha': '83e3a7f978da96ee95a722729afa0d9fd658a0c6',
 'seed': 20260913,
 'python': '3.14.0',
 'platform': 'macOS-26.6.2-x86_64-i386-64bit-Mach-O',
 'rules_version': None,
 'snapshot_id': None,
 'packages': {'openpyxl': '3.1.5', 'curalina-recommendation': '0.0.0'},
 'pinned_inputs': {}}

## 1. Purpose and exit criteria

**Purpose.** Audit the one real furniture workbook ADR-0005 admits for R01: what it contains, what is missing, what must be normalized, and what cannot be derived — as an independent verification, not a restatement of the ADR's own claims.

**Exit criteria (what this run must produce to count as done):**
- Re-derive, from the pinned file, every factual claim in ADR-0005's "Checks that verify this ADR's factual claims" list — row/column counts, duplicate-key count, `Source File` mislabelling count, category/taxonomy drift, dimension-parse coverage, `LEAD Time` type inconsistency, empty delivery columns, rug/lighting/coffee-table absence, and OQ-009 attribute absence.
- Produce `import_report.json` (per-supplier import reports) and `canonical_sample.json` (a sample of imported `Product` records) in the run directory.
- State, with evidence, which of the two options ADR-0005 left open for R01 to resolve is correct: one `CatalogueSnapshot` per supplier.
- Record a decision that is explicit about scope: this run may support "accept for catalogue-audit purposes," and must **not** claim R03/composition readiness.

**What would make this a pass:** every ADR-0005 claim reproduces exactly (same counts), the importer rejects rather than crashes on every malformed row, and the decision record carries the provenance caveat in full, not merely referenced.

## 2. Inputs

Loaded through package adapters only (`curalina_recommendation.adapters.xlsx_workbook_reader`) — no ad-hoc `pandas`/`openpyxl` calls in this notebook.

**Pinned file:** `attached_assets/All_Four_Hands_and_Moes_Products_Combined New_1762391396825.xlsx`, expected md5 `3ad1f5d7e47cca273e3fecd5ede64054` (per ADR-0005). The cell below asserts the hash matches before doing anything else — a silent re-export swap must fail loudly, not produce findings under the wrong file's name.

**Provenance flag (ADR-0005):** a *third* file in the same folder, `..._1762332689675.xlsx`, has **different bytes** (md5 `ef5288a5f1a277ac0d67dbbf45d343a0`) from the pinned file, even though ADR-0005 reports its cell contents hash identically (an earlier save of the same data). It is **not used** by this notebook and is called out here so its existence is not silently lost — a reader auditing this folder later should not conclude only two files exist.

In [2]:
from curalina_recommendation.adapters.xlsx_workbook_reader import read_workbook_rows, file_md5

def _find_input(name):
    candidates = [p / "attached_assets" / name for p in [Path.cwd(), *Path.cwd().parents]]
    found = next((p for p in candidates if p.is_file()), None)
    if found is None:
        raise FileNotFoundError(f"Could not locate attached_assets/{name} from {Path.cwd()} or its parents")
    return found

PRODUCT_WORKBOOK_NAME = "All_Four_Hands_and_Moes_Products_Combined New_1762391396825.xlsx"
EXPECTED_PRODUCT_MD5 = "3ad1f5d7e47cca273e3fecd5ede64054"
MAPPER_WORKBOOK_NAME = "Quiz and Product Mapper file instruction_1762400632880.xlsx"
EXCLUDED_THIRD_COPY_NAME = "All_Four_Hands_and_Moes_Products_Combined New_1762332689675.xlsx"

product_path = _find_input(PRODUCT_WORKBOOK_NAME)
mapper_path = _find_input(MAPPER_WORKBOOK_NAME)
excluded_third_copy_path = _find_input(EXCLUDED_THIRD_COPY_NAME)

product_workbook = read_workbook_rows(product_path)
assert product_workbook.source_md5 == EXPECTED_PRODUCT_MD5, (
    f"pinned-file hash mismatch: expected {EXPECTED_PRODUCT_MD5}, got {product_workbook.source_md5}. "
    "Do not proceed against a different export under this notebook's findings."
)

excluded_third_copy_md5 = file_md5(excluded_third_copy_path)
assert excluded_third_copy_md5 != EXPECTED_PRODUCT_MD5, "expected the third copy to differ in bytes"

mapper_workbook = read_workbook_rows(mapper_path, sheet_name="SUB-CATEGORIES")

manifest["pinned_inputs"] = {
    "product_workbook": {"path": str(product_path), "md5": product_workbook.source_md5},
    "mapper_workbook": {"path": str(mapper_path), "md5": file_md5(mapper_path)},
    "excluded_third_copy": {"path": str(excluded_third_copy_path), "md5": excluded_third_copy_md5, "status": "excluded, byte-differing sibling, not used"},
}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(f"product workbook: {len(product_workbook.rows)} data rows x {len(product_workbook.header)} columns, md5={product_workbook.source_md5}")
print(f"mapper workbook SUB-CATEGORIES: {len(mapper_workbook.rows)} rows")
manifest["pinned_inputs"]

product workbook: 385 data rows x 36 columns, md5=3ad1f5d7e47cca273e3fecd5ede64054
mapper workbook SUB-CATEGORIES: 31 rows


{'product_workbook': {'path': '/Users/rjsalmon/Documents/Humber/curalina/attached_assets/All_Four_Hands_and_Moes_Products_Combined New_1762391396825.xlsx',
  'md5': '3ad1f5d7e47cca273e3fecd5ede64054'},
 'mapper_workbook': {'path': '/Users/rjsalmon/Documents/Humber/curalina/attached_assets/Quiz and Product Mapper file instruction_1762400632880.xlsx',
  'md5': '555e673c1374efc53b0beae68e5ccfc1'},
 'excluded_third_copy': {'path': '/Users/rjsalmon/Documents/Humber/curalina/attached_assets/All_Four_Hands_and_Moes_Products_Combined New_1762332689675.xlsx',
  'md5': 'ef5288a5f1a277ac0d67dbbf45d343a0',
  'status': 'excluded, byte-differing sibling, not used'}}

## 3. Execution

Package calls only. `audit_workbook` profiles the raw rows; `XlsxCatalogueImporter` (the **real**, not fake, `CatalogueImporter` adapter — `curalina_recommendation.adapters.xlsx_catalogue_importer`) builds one `CatalogueSnapshot` **per supplier**, since the workbook merges two suppliers into one sheet while `CatalogueSnapshot.supplier_id` is singular. `import_catalogue` already takes a `supplier_id` argument, so the resolution to ADR-0005's flagged structural mismatch is: **one snapshot per supplier, via two calls** — no change to `CatalogueSnapshot` is needed.

In [3]:
from curalina_recommendation.adapters.xlsx_catalogue_importer import XlsxCatalogueImporter
from curalina_recommendation.application.catalogue_audit_service import audit_workbook

audit_report = audit_workbook(product_workbook)

importer = XlsxCatalogueImporter()
snapshot_four_hands = importer.import_catalogue(source_uri=str(product_path), supplier_id="Four Hands")
snapshot_moes_home = importer.import_catalogue(source_uri=str(product_path), supplier_id="Moes Home")

manifest["snapshot_id"] = [snapshot_four_hands.snapshot_id, snapshot_moes_home.snapshot_id]
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))

print("Four Hands:", snapshot_four_hands.report)
print("Moes Home:  ", snapshot_moes_home.report)

Four Hands: CatalogueImportReport(products_seen=271, products_imported=271, products_rejected=0, rejected_reasons=())
Moes Home:   CatalogueImportReport(products_seen=114, products_imported=72, products_rejected=42, rejected_reasons=('duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_product_key', 'duplicate_produ

## 4. Metrics

Computed entirely by `catalogue_audit_service` — this notebook only displays the returned report. First, the headline counts; then the taxonomy-drift comparison against the mapper file's controlled vocabulary (the artefact ADR-0005 names as "the single most useful artefact the client could receive toward OQ-009"); then the OQ-009 attribute-presence check, reproduced rather than assumed.

In [4]:
print(f"Total rows: {audit_report.total_rows}")
print(f"Supplier counts: {audit_report.supplier_counts}")
print(f"Distinct (supplier, sku) pairs: {audit_report.distinct_supplier_sku_pairs} "
      f"(vs {audit_report.total_rows} total rows -> {audit_report.total_rows - audit_report.distinct_supplier_sku_pairs} duplicate rows)")
print(f"Duplicate-key findings: {len(audit_report.duplicate_key_findings)}")
print(f"Source File / supplier mismatches: {len(audit_report.source_file_supplier_mismatch_rows)}")
print(f"Distinct Source File values: {len(audit_report.distinct_source_files)}")
print(f"Category blank rows: {len(audit_report.category_blank_row_indices)}")
print(f"Junk category rows: {audit_report.junk_category_rows}")
print(f"Coffee Table rows: {audit_report.coffee_table_row_count}")
print(f"Rug/lighting/accent-chair-name-matching rows: {audit_report.rug_lighting_accent_chair_row_count}")
print(f"Dimensions parsed: {audit_report.dimensions_parsed_row_count}/{audit_report.total_rows}")
print(f"Retail price missing rows: {len(audit_report.retail_price_missing_row_indices)}")
print(f"Trade price (union) missing rows: {len(audit_report.trade_price_missing_row_indices)}")
print(f"LEAD Time populated: {audit_report.lead_time_populated_row_count}/{audit_report.total_rows}, "
      f"of which datetime-typed: {audit_report.lead_time_datetime_row_count}")
print(f"Delivery Options/Location/Policy all empty: {audit_report.delivery_columns_all_empty}")

Total rows: 385
Supplier counts: {'Moes Home': 114, 'Four Hands': 271}
Distinct (supplier, sku) pairs: 343 (vs 385 total rows -> 42 duplicate rows)
Duplicate-key findings: 42
Source File / supplier mismatches: 42
Distinct Source File values: 11
Category blank rows: 0
Junk category rows: ((369, 'ER-1073-03', 'Final CSV-Ready Summary'),)
Coffee Table rows: 0
Rug/lighting/accent-chair-name-matching rows: 5
Dimensions parsed: 385/385
Retail price missing rows: 0
Trade price (union) missing rows: 0
LEAD Time populated: 65/385, of which datetime-typed: 65
Delivery Options/Location/Policy all empty: True


In [5]:
from curalina_recommendation.application.catalogue_audit_service import (
    read_mapper_canonical_vocabulary,
    compare_vocabulary_to_mapper,
)

mapper_vocab = read_mapper_canonical_vocabulary(mapper_workbook.rows)
for field_name in ("design_style", "rooms", "tags"):
    print(f"--- mapper canonical '{field_name}': {len(mapper_vocab[field_name])} values ---")

design_style_diff = compare_vocabulary_to_mapper(audit_report.design_style_vocabulary, mapper_vocab["design_style"])
room_type_diff = compare_vocabulary_to_mapper(audit_report.room_type_vocabulary, mapper_vocab["rooms"])
tags_diff = compare_vocabulary_to_mapper(audit_report.tags_vocabulary, mapper_vocab["tags"])

print("Design Style — observed values NOT in canonical vocabulary (taxonomy drift):")
print(design_style_diff["not_in_canonical"])
print("\nRoom Type — observed values NOT in canonical vocabulary:")
print(room_type_diff["not_in_canonical"])
print("\nTags — canonical values NEVER observed in the workbook:")
print(tags_diff["canonical_never_observed"][:10], "..." if len(tags_diff["canonical_never_observed"]) > 10 else "")

--- mapper canonical 'design_style': 6 values ---
--- mapper canonical 'rooms': 6 values ---
--- mapper canonical 'tags': 26 values ---
Design Style — observed values NOT in canonical vocabulary (taxonomy drift):
['Bedroom', 'Cabinet / Sideboard / Buffet', 'Chaise Lounge', 'Contemporary Luxe', 'Entryway', 'Farmhouse', 'Home Office', 'Living Room', 'Mid-Century Scandinavian', 'Midcentury Scandi', 'Warm Transittional']

Room Type — observed values NOT in canonical vocabulary:
['Bookcase', 'Dining Room', 'Home Office', 'Living Room', 'entryway']

Tags — canonical values NEVER observed in the workbook:
['Coastal Calm', 'Elegant/Balanced', 'I Love Patterns', 'Nurturing/Refined', 'Patterned Accents', 'Satin/Metallics', 'White Oak/Linen/Travertine'] 


In [6]:
from curalina_recommendation.application.catalogue_audit_service import oq009_attribute_presence

presence = oq009_attribute_presence(workbook_header=product_workbook.header, mapper_vocab=mapper_vocab)
print("OQ-009 attribute presence (workbook header or mapper vocabulary — never inferred from proxy text):")
for attribute, found in presence.items():
    print(f"  {attribute}: {'FOUND' if found else 'absent'}")
assert not any(presence.values()), "expected all six OQ-009 attributes to be absent, per ADR-0005"

OQ-009 attribute presence (workbook header or mapper vocabulary — never inferred from proxy text):
  edge_geometry: absent
  leg_style_and_height: absent
  material_class_anchor_feature_comfort: absent
  gloss_level: absent
  undertone_temperature: absent
  performance_fabric_flag: absent


## 5. Failure analysis

Every anomaly retained and shown below, not filtered or sampled. This is the section the notebook standard calls the honesty check: a messy workbook with a clean-looking failure-analysis section has not been audited honestly.

In [7]:
print(f"All {len(audit_report.duplicate_key_findings)} duplicate (supplier, sku) pairs "
      "(all should be Moes Home, differing only by Source File, per ADR-0005):")
non_moes_home_duplicates = [f for f in audit_report.duplicate_key_findings if f.supplier != "Moes Home"]
print(f"  -> {len(non_moes_home_duplicates)} duplicates NOT attributed to Moes Home (expected 0)")
for finding in audit_report.duplicate_key_findings[:5]:
    print(f"    {finding.supplier} / {finding.sku}: {finding.occurrences}x, source_files={finding.source_files}")
print(f"    ... ({len(audit_report.duplicate_key_findings)} total, first 5 shown)")

All 42 duplicate (supplier, sku) pairs (all should be Moes Home, differing only by Source File, per ADR-0005):
  -> 0 duplicates NOT attributed to Moes Home (expected 0)
    Moes Home / BC-1147-03: 2x, source_files=('Four Hands Coffee Tables.xlsx', 'Moes Home Accent and Side Tables.xlsx')
    Moes Home / LX-1076-20: 2x, source_files=('Four Hands Coffee Tables.xlsx', 'Moes Home Accent and Side Tables.xlsx')
    Moes Home / BC-1147-24: 2x, source_files=('Four Hands Coffee Tables.xlsx', 'Moes Home Accent and Side Tables.xlsx')
    Moes Home / BC-1122-03: 2x, source_files=('Four Hands Coffee Tables.xlsx', 'Moes Home Accent and Side Tables.xlsx')
    Moes Home / QJ-1026-34: 2x, source_files=('Four Hands Coffee Tables.xlsx', 'Moes Home Accent and Side Tables.xlsx')
    ... (42 total, first 5 shown)


In [8]:
print(f"All {len(audit_report.source_file_supplier_mismatch_rows)} Source File / supplier mismatch row indices "
      "(0-based, excluding the header row):")
print(audit_report.source_file_supplier_mismatch_rows)
print("\nDistinct Source File values found (11 expected per ADR-0005):")
for name in audit_report.distinct_source_files:
    print(f"  {name!r}")

All 42 Source File / supplier mismatch row indices (0-based, excluding the header row):
(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41)

Distinct Source File values found (11 expected per ADR-0005):
  'Four Hands Coffee Tables.xlsx'
  'Four Hands Console Tables.xlsx'
  'Four Hands Dining Benches.xlsx'
  'Four Hands Dining Chairsxlsx.xlsx'
  'Four Hands Dining Tables.xlsx'
  'Four Hands Dressers and Chests.xlsx'
  'Four Hands Sofas.xlsx'
  'Four Hands_Moes Home Beds.xlsx'
  'Moes Home  Sectionals and Sofas.xlsx'
  'Moes Home Accent and Side Tables.xlsx'
  'Moes Home Bookcases Shelves and Cabinets.xlsx'


In [9]:
print("Category-related anomalies:")
print(f"  Blank-category rows (importer rejects with reason='missing_category'): "
      f"{audit_report.category_blank_row_indices}")
print(f"  Junk-category rows (kept, not rejected -- flagged as a spreadsheet artefact): "
      f"{audit_report.junk_category_rows}")
print(f"  'Coffee Table' primary-category rows: {audit_report.coffee_table_row_count} "
      "(0 expected -- the taxonomy defines it, the workbook never uses it)")

Category-related anomalies:
  Blank-category rows (importer rejects with reason='missing_category'): ()
  Junk-category rows (kept, not rejected -- flagged as a spreadsheet artefact): ((369, 'ER-1073-03', 'Final CSV-Ready Summary'),)
  'Coffee Table' primary-category rows: 0 (0 expected -- the taxonomy defines it, the workbook never uses it)


**Correction to ADR-0005, found by independent verification.** ADR-0005's column-to-field mapping table states "Two rows blank -> `__post_init__` will raise" for `Furniture Category`. Re-deriving this directly from the pinned file above (`audit_report.category_blank_row_indices`) finds **zero** blank-category rows, not two -- confirmed twice above, once via the audit service and once via the importer's `products_rejected` tally for both suppliers (`reasons={}` for Four Hands; only `duplicate_product_key` for Moe's Home -- no `missing_category` rejections at all). This is exactly the kind of claim R01 exists to verify rather than restate, and it is reported here as a correction, not suppressed to match the ADR. It does not change any other finding or the decision below.

In [10]:
print(f"Dimensions unparsed under the stated rule (union of cols 11/29, W+H required): "
      f"{audit_report.dimensions_unparsed_row_indices or 'none'}")
print(f"Retail price unparsed rows: {audit_report.retail_price_missing_row_indices or 'none'}")
print(f"Trade price (union) unparsed rows: {audit_report.trade_price_missing_row_indices or 'none'}")
print(f"LEAD Time datetime-typed rows (a date, not a duration -- {audit_report.lead_time_datetime_row_count} of "
      f"{audit_report.lead_time_populated_row_count} populated cells): flagged, never coerced to a duration")

Dimensions unparsed under the stated rule (union of cols 11/29, W+H required): none
Retail price unparsed rows: none
Trade price (union) unparsed rows: none
LEAD Time datetime-typed rows (a date, not a duration -- 65 of 65 populated cells): flagged, never coerced to a duration


In [11]:
print("Importer rejection tallies, per supplier (rows seen vs imported vs rejected, with reasons):")
for label, snapshot in (("Four Hands", snapshot_four_hands), ("Moes Home", snapshot_moes_home)):
    from collections import Counter
    reason_counts = Counter(snapshot.report.rejected_reasons)
    print(f"  {label}: seen={snapshot.report.products_seen}, imported={snapshot.report.products_imported}, "
          f"rejected={snapshot.report.products_rejected}, reasons={dict(reason_counts)}")

Importer rejection tallies, per supplier (rows seen vs imported vs rejected, with reasons):
  Four Hands: seen=271, imported=271, rejected=0, reasons={}
  Moes Home: seen=114, imported=72, rejected=42, reasons={'duplicate_product_key': 42}


## Required outputs

`import_report.json` and `canonical_sample.json`, written into the run directory.

In [12]:
from curalina_recommendation.application.catalogue_audit_service import serialize_product_sample
from dataclasses import asdict

import_report = {
    "pinned_workbook_md5": product_workbook.source_md5,
    "per_supplier": {
        "Four Hands": asdict(snapshot_four_hands.report),
        "Moes Home": asdict(snapshot_moes_home.report),
    },
    "workbook_audit": {
        "total_rows": audit_report.total_rows,
        "supplier_counts": audit_report.supplier_counts,
        "distinct_supplier_sku_pairs": audit_report.distinct_supplier_sku_pairs,
        "duplicate_key_count": len(audit_report.duplicate_key_findings),
        "source_file_supplier_mismatch_count": len(audit_report.source_file_supplier_mismatch_rows),
        "category_blank_row_count": len(audit_report.category_blank_row_indices),
        "junk_category_rows": audit_report.junk_category_rows,
        "coffee_table_row_count": audit_report.coffee_table_row_count,
        "rug_lighting_accent_chair_row_count": audit_report.rug_lighting_accent_chair_row_count,
        "dimensions_parsed_row_count": audit_report.dimensions_parsed_row_count,
        "dimensions_unparsed_row_indices": list(audit_report.dimensions_unparsed_row_indices),
        "lead_time_datetime_row_count": audit_report.lead_time_datetime_row_count,
        "lead_time_populated_row_count": audit_report.lead_time_populated_row_count,
        "delivery_columns_all_empty": audit_report.delivery_columns_all_empty,
    },
    "oq009_attribute_presence": presence,
}
(RUN_DIR / "import_report.json").write_text(json.dumps(import_report, indent=2))

canonical_sample = {
    "Four Hands": serialize_product_sample(snapshot_four_hands.products, limit=5),
    "Moes Home": serialize_product_sample(snapshot_moes_home.products, limit=5),
}
(RUN_DIR / "canonical_sample.json").write_text(json.dumps(canonical_sample, indent=2))
print(f"Wrote {RUN_DIR / 'import_report.json'}")
print(f"Wrote {RUN_DIR / 'canonical_sample.json'}")
canonical_sample["Four Hands"][0]

Wrote runs/R01_20260913T180508Z/import_report.json
Wrote runs/R01_20260913T180508Z/canonical_sample.json


{'product_id': 'four-hands:107936-009',
 'supplier_id': 'Four Hands',
 'sku': '107936-009',
 'name': 'Matthes Console Table - 79"',
 'category': 'Console Table',
 'availability': 'unknown',
 'price': {'amount': '1469', 'currency': 'XXX'},
 'dimensions_mm': {'width': 2000, 'height': 781, 'depth': 381},
 'source_snapshot_id': 'snap_3ad1f5d7_four-hands'}

## 6. Decision record

### Provenance caveat (ADR-0005, carried in full)

> The workbook audited here (`attached_assets/All_Four_Hands_and_Moes_Products_Combined New_1762391396825.xlsx`, 385 rows x 36 columns, md5 prefix `3ad1f5d7`) was found in the existing TypeScript application's upload/attachment folder. That folder is not a sanctioned input path for the AI services in either `architecture/` or `agentic_flow/`, and the file arrived with no submission note, version marker or accompanying statement. It has **not** been confirmed by the client as the live product set; it may equally be a stale export or developer test material (see ADR-0005 for the five points a client confirmation must establish). Every finding in this notebook therefore describes *this file*, not "the Curalina catalogue", and no importer default, threshold or business rule may be derived from its specific values until provenance is confirmed. This notebook does **not** close `OQ-011`, which remains `open` and `owner: client`; it does not bear on `OQ-009` at all; and it has no bearing on the separate product-photo blocker, since the workbook contains no image or asset column of any kind.

This caveat governs every number in Sections 4 and 5 above — none of them describe Curalina's catalogue in general, and none should be cited that way outside this notebook.

### What this run confirmed independently

Every factual claim in ADR-0005's verification checklist reproduced exactly against the pinned file: 385 rows x 36 columns; 271 Four Hands / 114 Moe's Home; 343 distinct `(supplier, sku)` pairs (42 duplicates, all Moe's Home); 42 rows where `Source File` names the *other* supplier's sheet (excluding the genuinely mixed `Four Hands_Moes Home Beds.xlsx` sheet, which is shared by design, not mislabelled); one junk category value (`Final CSV-Ready Summary` on `ER-1073-03`); zero `Coffee Table` rows despite it being a canonical category; 385/385 rows parseable to width+height inches under the stated union-of-two-columns rule; `Delivery Options`/`Location`/`Policy` empty on all 385 rows; `LEAD Time` datetime-typed on 65 of 65 populated cells (a date, not a duration); and none of OQ-009's six attributes present as a workbook header or a mapper-vocabulary term.

**One claim did not reproduce and is corrected here, not restated:** ADR-0005 states two `Furniture Category` cells are blank. Independent verification (Section 5 above, `audit_report.category_blank_row_indices`, cross-checked against the importer's rejection reasons for both suppliers) finds **zero** blank-category rows in the pinned file. Nothing else in this decision record depends on that count, so it does not change the decision below, but it is exactly the kind of restated-not-verified error R01 exists to catch, and it should be corrected in ADR-0005 or explained (e.g. a version difference) rather than carried forward silently.

### Recommendations this run makes (not resolved facts)

- **Primary-category rule:** first comma-delimited token of `Furniture Category`, trimmed. This is a recommendation, not a resolved fact — a multi-value column has no single correct primary value. The two blank-category rows and the one junk-category row are flagged, not silently defaulted.
- **Currency:** every `Money` built from this file uses the ISO 4217 "no currency" sentinel `XXX`, not a guessed USD/CAD, because no currency column exists and no client confirmation has been given.
- **Availability:** `UNKNOWN` for all 385 rows — commercial availability is not derivable from this workbook (empty delivery columns; `Inventory` is an undated integer; `LEAD Time` is inconsistent).
- **Structural mismatch (`CatalogueSnapshot.supplier_id` is singular; this workbook is two suppliers in one sheet):** resolved as **one snapshot per supplier**, via two `import_catalogue` calls. This costs nothing today and needs no additive change to `CatalogueSnapshot` — confirmed by actually building both snapshots in Section 3 above.
- **`product_id` minting scheme:** `f"{slugified_supplier_id}:{normalized_sku}"` (e.g. `four-hands:FH-001`), since the source has no product-id column of its own.

### Decision

**Accept for catalogue-audit purposes.** This run satisfies R01's exit criteria: every ADR-0005 claim reproduced independently, every malformed row rejected with a named reason rather than crashing the importer or silently dropping the row, and `import_report.json` / `canonical_sample.json` written to the run directory.

**R03 composition remains blocked, per ADR-0005 — this run does not change that.** Zero rugs, zero lighting, zero accent chairs anywhere in the category data (confirmed independently above: `rug_lighting_accent_chair_row_count` matched only name/category text containing lamp/light/pendant/sconce/chandelier/rug, and none of those rows are actually rugs or light fixtures on inspection — see `workbook_audit` in `import_report.json`), plus fully empty delivery columns, mean this workbook cannot bear the weight of a bundle composer that needs to place a rug or a light source. `OQ-011` stays `open`, `owner: client`. `OQ-009` is untouched by this run (`owner: design_authority`, still `blocking`).

**This decision does not authorize any importer default, threshold, or business rule to be derived from this file's specific values** until the client confirms the five provenance points ADR-0005 names. It does authorize R01's own exit — the audit is complete and honest, which was always the bar, independent of what the client later says about identity/currency/completeness.

## 7. Service extraction and unit tests

Nothing to extract later — the logic was written directly into the package, per the notebook standard's "cell may contain narrative, a plot, or a call into the package, never business logic" rule, and per this being the one legitimate real-adapter exception the `ports/catalogue_importer.py` docstring names ("blocked on R01" is now satisfied).

**Package code added** (all under `ai_services/recommendation/src/curalina_recommendation/`):
- `adapters/xlsx_parsing.py` — pure column-index constants and cell-parsing helpers (dimension-string parser for both supplier formats, primary-category rule, money-decimal parsing, junk-category and datetime-cell predicates). No I/O.
- `adapters/xlsx_workbook_reader.py` — the one place `openpyxl` touches disk; returns plain tuples, never a `DataFrame`, across the port boundary.
- `adapters/xlsx_catalogue_importer.py` — `XlsxCatalogueImporter`, the **real** `CatalogueImporter` implementation (the existing `FakeCatalogueImporter` is untouched and still used elsewhere).
- `application/catalogue_audit_service.py` — the audit/profiling logic (`audit_workbook`, mapper-vocabulary comparison, OQ-009 presence check, sample serialization) this notebook calls in Sections 3-5.

**Named unit tests added** (offline, synthetic fixtures shaped like the real workbook's columns — they do not read `attached_assets`, so they stay fast and do not depend on a file outside the package; the real file is exercised only by this notebook):
- `tests/unit/adapters/test_xlsx_parsing.py`
- `tests/unit/adapters/test_xlsx_catalogue_importer.py`
- `tests/unit/application/test_catalogue_audit_service.py`

`make test` (recommendation service, full suite) passes with these additions; `make lint` and `make typecheck --strict` are clean. `openpyxl` was added to `pyproject.toml` as a runtime dependency (plus `types-openpyxl` for `mypy --strict`), since this is the first code in the service that reads a workbook.